# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.3.3


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [ ]:
def clean_export(source):
    #копия, чтобы не менять исходную таблицу
    work = source.copy(deep=True)
    
        #known_key: строки с известным source_row
    known_key = work["source_row"].notna()
    # Среди известных ключей убираем полностью одинаковые строки;
    # если после этого ключ всё ещё повторяется, значит есть конфликт версий -> остановка
    unique_work = work.loc[known_key].drop_duplicates()
    assert unique_work["source_row"].is_unique, "у одного source_row разные версии"
    duplicates_mask = known_key & work["source_row"].duplicated()
    work = work.loc[~duplicates_mask].copy()
    duplicates_num = int(duplicates_mask.sum())

        #Текстовая серия для всех оставшихся строк; "" означает "нарушений нет"
    reasons = pd.Series("", index = work.index, dtype="string")

        #Сначала убираем внешние пробелы, затем проверяем код; пропуск тоже считается нарушением
    work["county_code"] = work["county_code"].str.strip()
    wrong_county = ~work["county_code"].eq("11001").fillna(False)
    reasons.loc[wrong_county] += "wrong_county;"

        #преобразование source_row, action_taken и loan_amount в число, если не проходит - меняем неподходящее поле на n/a и считаем ошибки в numeric_errors
    numeric_errors = {}
    for row in ["source_row", "action_taken", "loan_amount", "income"]:
        numeric = pd.to_numeric(work[row], errors="coerce")
        numeric_errors[row] = int((work[row].notna() & numeric.isna()).sum())
        work[row] = numeric
    reasons.loc[work["source_row"].isna()] += "wrong_source_row;"

        #Допустимы только коды 1, 2, 3 (NA в isin даёт False, значит строка в карантин)
    inside_123_mask = work["action_taken"].isin([1,2,3])
    reasons.loc[~inside_123_mask] += "wrong_action_taken;"

        #проверяю, что сумма заполнена и не равна 0, если условие не выполняется - в карантин
    wrong_amount_mask = work["loan_amount"].isna() | work["loan_amount"].le(0)
    reasons.loc[wrong_amount_mask] += "wrong_loan_amount;"

        #создаю таблицу wrong, где остаются только карантинные строки
    wrong = reasons.ne("")
        #создаю очищенную таблиу
    clean = work.loc[~wrong].copy()
        #создаю таблицу карантина
    quarantine = source.loc[work.index[wrong]].copy()
    quarantine["reason"] = reasons.loc[wrong]
        #создаю словарь с балансом
    audit = {
        "input": len(source), "replays": duplicates_num,
        "clean": len(clean), "quarantine": len(quarantine),
        "parse_errors": numeric_errors,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"]
    return clean, quarantine, audit


print(quarantine, "\n", audit)


  source_row activity_year                   lei county_code action_taken  \
0          2          2023  549300VZVN841I2ILS84       11001            1   
1          3          2023  549300P1347BFCBHJV60       11001           88   

  loan_amount income               reason  
0        oops   72.0   wrong_loan_amount;  
1    585000.0  174.0  wrong_action_taken;   
 {'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'action_taken': 0, 'loan_amount': 1, 'income': 1}}


## H2. Год и известный доход

In [ ]:
clean, quarantine, audit = clean_export(export)

assert clean["activity_year"].eq("2023").fillna(False).all(), "В activity_year некорректное значение"
total = len(clean)

initial_income = clean["income"]
na_income = clean["income"].isna()
known_income = clean["income"].notna()

print("Общее количество записей income: ", len(initial_income), "\n")
print("поля income с отсутствующими значениями ", na_income.sum(), "\n") #тут добавится 1 строка, где было BAD
print("Поля income с известными значениями: ", known_income.sum(), "\n")


print("Средний известный доход: ", known_income_num.mean(), "тыс. USD") #.mean() игнорирует NA, поэтому считаю средний известный доход по clean

assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"], "Баланс расходится"
assert total == audit["clean"] "Таблица изменилась"
assert_frame_equal(export, raw_copy)   # export не изменился
print("Баланс сходится, export не изменился")

Общее количество записей income:  6828 

поля income с отсутствующими значениями  343 

Поля income с известными значениями:  6485 

Средний известный доход:  237.06553585196608 тыс. USD
Баланс сходится, export не изменился


## H3. Проверка отчётного года

In [133]:
wrong_year = export.copy(deep=True)

wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

changed = wrong_year["activity_year"].ne(export["activity_year"]).sum()
assert changed == 1
assert wrong_year.drop(columns="activity_year").equals(export.drop(columns="activity_year")) #идентичны, за исключением activity_year

year_match = wrong_year["activity_year"].eq("2023").fillna(False)
year_ok = year_match.all()

print("year_ok:", year_ok)
print("Строк с годом не 2023:", int((~year_match).sum()))

assert not year_ok, "Все значения равны 2023(а так быть не должно)"

assert_frame_equal(export, raw_copy)
print("export сохранился")

year_ok: False
Строк с годом не 2023: 1
export сохранился


## H+1. Конфликт исходных значений

In [134]:
conflict = export.copy(deep=True)

replay_rows = conflict["source_row"].notna() & conflict["source_row"].duplicated()
last_replay = conflict.index[replay_rows][-1]      # индекс 6831 (source_row 16)

conflict.loc[last_replay, "loan_amount"] = "1.0"

try:
    clean_export(conflict)
except AssertionError as e:
    print("Ожидаемая остановка:", e)
else:
    raise RuntimeError("Ошибка проверки: конфликт версий не обнаружен")

assert_frame_equal(export, raw_copy)

Ожидаемая остановка: В source_row есть повторяющиеся значения


## H+2. Совпадение опубликованных полей

In [ ]:
fields = [c for c in decisions.columns if c != "source_row"] #source_row всегда уникален

dedup = decisions.drop_duplicates(subset=fields)

print("Строк до:", len(decisions), "после:", len(dedup),
      "разница:", len(decisions) - len(dedup))
print("Сумма loan_amount до:", decisions["loan_amount"].sum(),
      "после:", dedup["loan_amount"].sum(),
      "разница:", decisions["loan_amount"].sum() - dedup["loan_amount"].sum())

Строк до: 6830 после: 6757 разница: 73
Сумма loan_amount до: 4577460000.0 после: 4547855000.0 разница: 29605000.0
